# Stage5:无 LoRA 基座 —— 两种注意力机制的"原生"影响对比(n_cond=1)

**为什么要去掉 LoRA**:stage4 用的自训 feature_reuse LoRA 是 `independent_condition: true` 训出来的,
默认模式对它是分布外推理 —— 对比天然偏向 KV-Cache 一侧,不公平。
去掉 LoRA 后,两种模式对裸 FLUX.1-dev **同样都是分布外**,起点公平,
比的是注意力机制本身对生成的原生影响:

| | 默认模式 | 独立条件模式 |
|---|---|---|
| `kv_cache` | `False` | `True` |
| 条件分支 | 每步重算,与主图双向联合注意力 | 第 0 步算一次,之后主图单向读缓存 |
| LoRA | **无**(adapter=None,纯基座权重) | 同左 |
| seed / prompt / steps / 噪声起点 | 完全相同 | 同左 |

**预期口径**:裸基座没学过"条件 token 是控制信号",两列图大概率都不贴合 canny 结构
(这不是 bug,是实验设定)。要看的是:条件 token 作为"异物"注入后,
哪种交互方式对主图的干扰更大 —— 伪影、色偏、结构崩坏,以及随 TARGET_SIZE
(条件 token 占比)如何变化。

**用法**:只改 cell 1 的 `TARGET_SIZE`(256 / 512 / 1024),输出目录自动切换为
`repro/base_mode_compare_<TARGET_SIZE>/`,互不覆盖;最终产物 `comparison_grid.png`
与 stage4 同款([prompt | 条件图 | 默认 | KV-Cache] 网格)。

In [1]:
import os
os.chdir("..")  # 仓库根目录(与其它 repro notebook 一致)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import sys, json, time, random
sys.path.insert(0, "."); sys.path.insert(0, "repro")

import torch
# 与 stage3/probe 相同的兜底:torch 2.8 + diffusers 0.38 的 VAE conv_in 问题
torch.backends.cudnn.enabled = False

from PIL import Image
from kvcache_benchmark import load_pipeline
from omini.pipeline.flux_omini import Condition, convert_to_condition, generate

# ── 唯一需要改的开关:生成分辨率 ──────────────────────────────────────
TARGET_SIZE = 1024                       # 256 / 512 / 1024,输出目录自动跟随
# ──────────────────────────────────────────────────────────────────
COND_SIZE   = 256                       # 条件图固定 256,与 stage4 可比
POS_SCALE   = TARGET_SIZE / COND_SIZE   # 位置编码网格必须拉伸对齐,否则错位
STEPS       = 28                        # dev 质量档
SEEDS       = [42, 1234]
OUT_DIR     = f"repro/base_mode_compare_{TARGET_SIZE}"  # 随 TARGET_SIZE 切换,多档并存
os.makedirs(OUT_DIR, exist_ok=True)
os.environ["OUT_DIR"] = OUT_DIR   # 同步给可视化 cell 的 shell magic 用
print(f"[DEBUG] setup: 无LoRA steps={STEPS} cond={COND_SIZE}px target={TARGET_SIZE}px out={OUT_DIR}")

16:08:08 [INFO] cudnn disabled (workaround for VAE CUDNN_STATUS_NOT_INITIALIZED)
16:08:11 [WARNING] Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.8.0+cu128).
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


[DEBUG] setup: 无LoRA steps=28 cond=256px target=1024px out=repro/base_mode_compare_1024


In [2]:
# 强制走本地 HF cache + 直接用 snapshot 路径(双保险,完全绕开网络)
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

LOCAL_FLUX_PATH = "/home/wuwenxuan03/.cache/huggingface/hub/models--black-forest-labs--FLUX.1-dev/snapshots/3de623fc3c33e44ffbe2bad470d0f45bccf2eb21"
print("LOCAL_FLUX_PATH exists:", os.path.isdir(LOCAL_FLUX_PATH))

LOCAL_FLUX_PATH exists: True


In [3]:
# 加载 pipeline(dispatch=auto → 24G 卡走 2gpu 拆分)。
# 关键差异 vs stage4:不调 attach_lora —— 纯基座。
# adapter=None 时 lora_forward 直接走原始权重(flux_omini.py 的 None 分支),无需任何 PEFT 状态。
pipe = load_pipeline(LOCAL_FLUX_PATH, dispatch="auto")

16:08:12 [INFO] GPU 0: NVIDIA GeForce RTX 4090 | 23.6 GB
16:08:12 [INFO] GPU 1: NVIDIA GeForce RTX 4090 | 23.6 GB
16:08:12 [INFO] GPU 2: NVIDIA GeForce RTX 4090 | 23.6 GB
16:08:12 [INFO] GPU 3: NVIDIA GeForce RTX 4090 | 23.6 GB
16:08:12 [INFO] GPU 4: NVIDIA GeForce RTX 4090 | 23.6 GB
16:08:12 [INFO] GPU 5: NVIDIA GeForce RTX 4090 | 23.6 GB
16:08:12 [INFO] GPU 6: NVIDIA GeForce RTX 4090 | 23.6 GB
16:08:12 [INFO] GPU 7: NVIDIA GeForce RTX 4090 | 23.6 GB
16:08:12 [INFO] dispatch=auto -> 选择 2gpu(GPU0 24 GB)
16:08:12 [INFO] 加载 FLUX pipeline: /home/wuwenxuan03/.cache/huggingface/hub/models--black-forest-labs--FLUX.1-dev/snapshots/3de623fc3c33e44ffbe2bad470d0f45bccf2eb21 (bf16, 2gpu 手工 dispatch)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
16:09:52 [INFO] dispatch 完成: cuda:0 = encoders+vae+latents(主场), cuda:1 = transformer
16:09:52 [INFO] 已安装 tx bridge:输入 -> transformer 卡,输出 -> 主场卡


In [4]:
# 与 stage4 完全相同的 5 个 case,保证跨实验可比
CASES = [
    ("vase",    "assets/vase_hq.jpg",     "A beautiful ceramic vase with flowers on a wooden table, soft light."),
    ("room",    "assets/room_corner.jpg", "A cozy room corner with an armchair and warm afternoon lighting."),
    ("oranges", "assets/oranges.jpg",     "Fresh ripe oranges on a plate, studio product photography."),
    ("clock",   "assets/clock.jpg",       "A vintage alarm clock on a desk, detailed, photorealistic."),
    ("rc_car",  "assets/rc_car.jpg",      "A red remote control toy car on the ground, sharp focus."),
]

def build_condition(image_path):
    img = Image.open(image_path).convert("RGB").resize((COND_SIZE, COND_SIZE))
    cond_img = convert_to_condition("canny", img)
    # adapter_setting=None:条件分支用基座权重跑,不挂任何 LoRA —— 本实验的核心设定
    return cond_img, Condition(cond_img, None, position_scale=POS_SCALE)

def run_one(prompt, condition, seed, kv_cache):
    # WHY 每次重建 generator:两种模式必须从同一噪声出发,差异才全部归因于注意力模式
    g = torch.Generator(device="cuda:0").manual_seed(seed)
    t0 = time.perf_counter()
    out = generate(pipe, prompt=prompt, conditions=[condition],
                   height=TARGET_SIZE, width=TARGET_SIZE,
                   num_inference_steps=STEPS, guidance_scale=3.5,
                   generator=g, kv_cache=kv_cache)
    return out.images[0], time.perf_counter() - t0

In [5]:
# 主循环:5 case × 2 seed × 2 模式 = 20 次生成
records = []
for name, path, prompt in CASES:
    cond_img, condition = build_condition(path)
    cond_img.save(f"{OUT_DIR}/{name}_cond.png")
    for seed in SEEDS:
        row = {"case": name, "seed": seed, "prompt": prompt}
        for mode, kv in (("default", False), ("kvcache", True)):
            img, dt = run_one(prompt, condition, seed, kv)
            fp = f"{OUT_DIR}/{name}_s{seed}_{mode}.png"
            img.save(fp)
            row[mode] = fp; row[f"{mode}_sec"] = round(dt, 1)
        records.append(row)
        print(f"[DEBUG] run_one: {name} seed={seed} "
              f"default={row['default_sec']}s kvcache={row['kvcache_sec']}s "
              f"speedup={row['default_sec']/row['kvcache_sec']:.2f}x")

with open(f"{OUT_DIR}/records.json", "w") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f"完成 {len(records)} 组对照,已写 {OUT_DIR}/records.json")

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

[DEBUG] run_one: vase seed=42 default=20.6s kvcache=18.4s speedup=1.12x


  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

...

  0%|          | 0/28 [00:00<?, ?it/s]

[DEBUG] run_one: rc_car seed=42 default=20.2s kvcache=18.6s speedup=1.09x


  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

[DEBUG] run_one: rc_car seed=1234 default=20.3s kvcache=18.6s speedup=1.09x
完成 10 组对照,已写 repro/base_mode_compare_1024/records.json


In [6]:
# 可视化:复用 stage4 的独立脚本,产物 $OUT_DIR/comparison_grid.png(同款网格)
!python repro/visualize_quality_compare.py \
    --records $OUT_DIR/records.json \
    --out-dir $OUT_DIR

[INFO] repo_root  : /home/wuwenxuan03/OminiControl
[INFO] records    : repro/base_mode_compare_1024/records.json
[INFO] out_dir    : repro/base_mode_compare_1024
[INFO] 已生成     : repro/base_mode_compare_1024/comparison_grid.png  (4525 KB)

对比摘要  (10 组, 同 LoRA 同 seed)
  vase     s  42  default= 20.6s  kvcache= 18.4s  speedup=1.12x
  vase     s1234  default= 20.2s  kvcache= 18.5s  speedup=1.09x
  room     s  42  default= 20.3s  kvcache= 18.6s  speedup=1.09x
  room     s1234  default= 20.3s  kvcache= 18.6s  speedup=1.09x
  oranges  s  42  default= 20.3s  kvcache= 18.6s  speedup=1.09x
  oranges  s1234  default= 20.3s  kvcache= 18.6s  speedup=1.09x
  clock    s  42  default= 20.3s  kvcache= 18.5s  speedup=1.10x
  clock    s1234  default= 20.2s  kvcache= 18.6s  speedup=1.09x
  rc_car   s  42  default= 20.2s  kvcache= 18.6s  speedup=1.09x
  rc_car   s1234  default= 20.3s  kvcache= 18.6s  speedup=1.09x
  -> 平均 speedup: 1.09x


In [ ]:
# GSB 标注骨架(格式对齐 scripts/compute_gsb.py),盲评随机换边
rng = random.Random(0)
annotations, side_map = {}, {}
for r in records:
    cid = f"{r['case']}_s{r['seed']}"
    flip = rng.random() < 0.5
    side_map[cid] = {"A": "kvcache" if flip else "default",
                     "B": "default" if flip else "kvcache"}
    annotations[cid] = {"prompt": r["prompt"], "choices": {"quality": ""}}

with open(f"{OUT_DIR}/annotations.json", "w") as f:
    json.dump({"annotations": annotations}, f, ensure_ascii=False, indent=2)
with open(f"{OUT_DIR}/side_map.json", "w") as f:
    json.dump(side_map, f, ensure_ascii=False, indent=2)
print(f"标注后统计: python scripts/compute_gsb.py {OUT_DIR}/annotations.json")

## 读图指引

- **别用"贴不贴合 canny"打分** —— 裸基座本来就不会遵循条件,两列都不贴是正常的。
- 该看的是**主图被条件 token 干扰的程度**:伪影 / 噪斑 / 色偏 / 构图崩坏。
  默认模式下条件 token 每步与主图双向交互(干扰通道常开);独立模式下只在第 0 步
  编码一次、主图单向读取(干扰是静态的)。
- 跑 256 / 512 / 1024 三档后横向看:条件 token 占序列比例分别为 1/2、1/5、1/17,
  若默认模式的劣化随占比升高而加重,即可把 stage4 里 256 档的 default 伪影
  归因到"交互机制 × 条件占比",而不是 LoRA 训练分布。
- 与 stage4 同目录结构、同 case 同 seed,可直接并排对比"有 LoRA vs 无 LoRA"。